# Leson 11 - Agent-to-Agent (A2A) Protokol


## How to set am up


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin be A2A Protocol?

Di **Agent-to-Agent (A2A) protocol** na open standard wey enable AI agents to communicate,
find each oda, and collaborate — even if dem dey built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents dey publish an *Agent Card* wey dey describe their capabilities, making am
  easy for other agents (or orchestrators) to find di right specialist for a task.
- **Message Passing** – Agents dey exchange structured messages through a common protocol, so a
  request from one agent fit be understood and fulfilled by another regardless of e internal
  implementation.
- **Task Lifecycle** – A2A dey define states such as *submitted*, *working*, *completed*, and
  *failed*, wey dey give the orchestrator full visibility into how a delegated task dey progress.

For dis lesson we go simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent go contribute im expertise and pass results to the next.


## Di way wey you fit make special travel agent dem


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## How Multiple Agents Dey Collaborate via Workflow

We join the three agents into a sequential workflow wey dey mirror A2A message passing:

1. **CurrencyExchangeAgent** dey receive di user request an produce currency guidance.
2. **ActivityPlannerAgent** dey receive di enriched context an add activity recommendations.
3. **TravelManagerAgent** dey combine both inputs into di final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## How A2A Dey Work for Production

For production environment, di A2A protocol dey unlock powerful cross-service scenarios:

| Wetin e fit do | Wetin e mean |
|---|---|
| **How different frameworks dem dey work together** | One agent wey dem build with one framework fit give tasks to agent wey dem build with any oda A2A-compliant framework, so organisations fit interoperate for real. |
| **Service boundaries** | Agents fit dey for separate microservices, cloud regions, or even different organisations, but dem go still collaborate smoothly. |
| **How e dey find specialists on the fly** | Orchestrator fit query an Agent Card registry at runtime to find the specialist wey best for the sub-task. |
| **Streaming & push notifications** | A2A dey support Server-Sent Events (SSE) for real-time progress updates and push notifications for long-running tasks. |

Di workflow wey we build above na simplified, in-process version of dis pattern. For real
deployment each agent go expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Summary

For dis lesson you don learn:

1. **Wetin the A2A protocol be** — na open standard wey dey for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — di Currency Exchange agent, di Activity Planner agent, and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model one-after-anoda
   as messages dey pass between agents.
4. **How A2A works in production** — e dey enable cross-framework, cross-service collaboration
   wey get dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Abeg note:
Dem don translate dis document with AI translation service (Co-op Translator: https://github.com/Azure/co-op-translator). Even though we dey try make am correct, make you sabi say automatic translations fit get mistakes or no too correct. Di original document for im own language na im be di official source. If na important information, e better make professional human translator do am. We no go take any responsibility for any misunderstanding or wrong interpretation wey fit come from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
